# Лабораторная работа 1 — рабочий ноутбук

**ФИО:**
**Файлы данных:**

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

# Пути к файлам вашего варианта S32
RAW_CLIENTS = os.path.join("data", "S32", "clients.csv")
RAW_TXN = os.path.join("data", "S32", "transactions.csv")

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [27]:
import csv

def detect_file_params(filepath):
    """Определяет кодировку и разделитель файла."""
    # 1. Проверяем популярные кодировки
    encodings = ['utf-8', 'windows-1251', 'cp1251', 'utf-8-sig']
    correct_enc = None
    sample_text = ""
    
    for enc in encodings:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                sample_text = "".join([f.readline() for _ in range(15)])
                correct_enc = enc
                break
        except (UnicodeDecodeError, UnicodeError):
            continue
            
    # 2. Определяем разделитель колонок
    sniffer = csv.Sniffer()
    try:
        dialect = sniffer.sniff(sample_text)
        delimiter = dialect.delimiter
    except Exception:
        # Резервная эвристика, если sniffer не справился
        first_line = sample_text.split('\n')[0]
        delimiter = ';' if ';' in first_line else (',' if ',' in first_line else '\t')
        
    return correct_enc, delimiter

# Исследуем параметры файлов
clients_enc, clients_sep = detect_file_params(RAW_CLIENTS)
txn_enc, txn_sep = detect_file_params(RAW_TXN)

print(f"Параметры {RAW_CLIENTS}: кодировка={clients_enc}, разделитель='{clients_sep}'")
print(f"Параметры {RAW_TXN}: кодировка={txn_enc}, разделитель='{txn_sep}'")

# Загружаем сырые таблицы
# Пока не форсируем типы и парсинг дат, чтобы в разделе аудита увидеть исходные данные
df_clients_raw = pd.read_csv(RAW_CLIENTS, encoding=clients_enc, sep=clients_sep)
df_txn_raw = pd.read_csv(RAW_TXN, encoding=txn_enc, sep=txn_sep)

print("\n=== clients.csv: первые 3 строки ===")
display(df_clients_raw.head(3))
print(f"Размер: {df_clients_raw.shape[0]} строк, {df_clients_raw.shape[1]} колонок")

print("\n=== transactions.csv: первые 3 строки ===")
display(df_txn_raw.head(3))
print(f"Размер: {df_txn_raw.shape[0]} строк, {df_txn_raw.shape[1]} колонок")

Параметры data\S32\clients.csv: кодировка=utf-8, разделитель=','
Параметры data\S32\transactions.csv: кодировка=windows-1251, разделитель=';'

=== clients.csv: первые 3 строки ===


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C000864,1959-02-13,М,Нижний Новгород,Массовый,Серебро,Дебетовая,11.02.2021,29.07.2024,Не указан,37200.0,Да,2024-12,NaN,300,Нет
1,C000608,1980-05-27,М,Санкт-Петербург,Массовый,Серебро,Дебетовая,08.06.2012,19.05.2018,Не указан,51400.0,Да,2024-06,NaN,131,Да
2,C005220,1963-07-31,М,Санкт Петербург,Зарплатный,Золото,Дебетовая,15.02.2023,26.07.2024,Зарплата в нашем банке,115200.0,Да,2025-03,5.0,845,Нет


Размер: 5693 строк, 16 колонок

=== transactions.csv: первые 3 строки ===


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990147,C001784,2023-12-01 05:36:36,"439,6",RUB,Связь и интернет,4814,СБП,успешно
1,990049,C002958,2023-12-04 11:32:23,"48461,62",RUB,Одежда и обувь,5651,Карта,успешно
2,990214,C002389,2023-12-10 05:52:56,"763,12",RUB,Связь и Интернет,4814,Онлайн-банк,успешно


Размер: 628754 строк, 9 колонок


## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [28]:
print("=== 2.1. ОБЪЁМ И СТРУКТУРА ДАННЫХ ===")
print(f"Клиенты (clients.csv): {df_clients_raw.shape[0]} строк, {df_clients_raw.shape[1]} колонок")
print(f"Транзакции (transactions.csv): {df_txn_raw.shape[0]} строк, {df_txn_raw.shape[1]} колонок\n")

print("--- Типы данных clients.csv ---")
print(df_clients_raw.dtypes)

print("\n--- Типы данных transactions.csv ---")
print(df_txn_raw.dtypes)

# Проверка уникальности первичных ключей (Primary Keys)
client_pk_unique = df_clients_raw['client_id'].nunique()
client_pk_total = len(df_clients_raw)
print(f"\nУникальность client_id в clients: {client_pk_unique} из {client_pk_total} (дублей: {client_pk_total - client_pk_unique})")

txn_pk_unique = df_txn_raw['txn_id'].nunique()
txn_pk_total = len(df_txn_raw)
print(f"Уникальность txn_id в transactions: {txn_pk_unique} из {txn_pk_total} (дублей: {txn_pk_total - txn_pk_unique})")

print("\nЕдиница наблюдения:")
print("- Таблица clients: 1 строка = 1 уникальный действующий клиент банка на 30.06.2025")
print("- Таблица transactions: 1 строка = 1 операция по банковской карте")

=== 2.1. ОБЪЁМ И СТРУКТУРА ДАННЫХ ===
Клиенты (clients.csv): 5693 строк, 16 колонок
Транзакции (transactions.csv): 628754 строк, 9 колонок

--- Типы данных clients.csv ---
client_id              object
birth_date             object
gender                 object
city                   object
segment                object
loyalty_tier           object
card_type              object
registration_date      object
card_open_date         object
income_source          object
monthly_income        float64
has_mobile_app         object
app_last_login         object
satisfaction_score    float64
branch_code             int64
marketing_opt_in       object
dtype: object

--- Типы данных transactions.csv ---
txn_id           int64
client_id       object
txn_datetime    object
amount          object
currency        object
category        object
mcc_code         int64
channel         object
status          object
dtype: object

Уникальность client_id в clients: 5638 из 5693 (дублей: 55)
Уникальность t

### 2.2. Дубли

In [29]:
print("=== 2.2. АУДИТ ДУБЛИКАТОВ ===")

# 1. Полные дубликаты строк
full_dup_clients = df_clients_raw.duplicated().sum()
full_dup_txn = df_txn_raw.duplicated().sum()

print(f"Полные дубли в clients: {full_dup_clients} из {len(df_clients_raw)} строк ({full_dup_clients/len(df_clients_raw):.4%})")
print(f"Полные дубли в transactions: {full_dup_txn} из {len(df_txn_raw)} строк ({full_dup_txn/len(df_txn_raw):.4%})")

# 2. Дубликаты по ключевым полям (ID)
key_dup_clients = df_clients_raw.duplicated(subset=['client_id']).sum()
key_dup_txn = df_txn_raw.duplicated(subset=['txn_id']).sum()

print(f"\nДубли по client_id в clients: {key_dup_clients} из {len(df_clients_raw)} ({key_dup_clients/len(df_clients_raw):.4%})")
print(f"Дубли по txn_id в transactions: {key_dup_txn} из {len(df_txn_raw)} ({key_dup_txn/len(df_txn_raw):.4%})")
print("=" * 70)
print("=== ДЕТАЛЬНЫЙ АНАЛИЗ НЕПОЛНЫХ ДУБЛИКАТОВ ПО ID ===")
print("=" * 70)

# 1. Анализ clients.csv
# Отбрасываем абсолютно полные дубли, чтобы остались только конфликтующие
clients_no_exact_dups = df_clients_raw.drop_duplicates()
conflicting_clients = clients_no_exact_dups[clients_no_exact_dups.duplicated(subset=['client_id'], keep=False)]

print(f"\n[clients.csv] Строк с конфликтующими client_id: {len(conflicting_clients)}")
print(f"Уникальных client_id среди них: {conflicting_clients['client_id'].nunique()}")

if len(conflicting_clients) > 0:
    print("\nПримеры конфликтующих записей клиентов (сравните строки с одинаковым client_id):")
    # Сортируем по ID, чтобы строки одного клиента шли рядом
    display(conflicting_clients.sort_values(by='client_id'))

# 2. Анализ transactions.csv
txn_no_exact_dups = df_txn_raw.drop_duplicates()
conflicting_txns = txn_no_exact_dups[txn_no_exact_dups.duplicated(subset=['txn_id'], keep=False)]

print(f"\n[transactions.csv] Строк с конфликтующими txn_id: {len(conflicting_txns)}")
print(f"Уникальных txn_id среди них: {conflicting_txns['txn_id'].nunique()}")

if len(conflicting_txns) > 0:
    print("\nПримеры конфликтующих транзакций (первые 6 строк):")
    display(conflicting_txns.sort_values(by='txn_id').head(26))

=== 2.2. АУДИТ ДУБЛИКАТОВ ===
Полные дубли в clients: 42 из 5693 строк (0.7377%)
Полные дубли в transactions: 1279 из 628754 строк (0.2034%)

Дубли по client_id в clients: 55 из 5693 (0.9661%)
Дубли по txn_id в transactions: 1500 из 628754 (0.2386%)
=== ДЕТАЛЬНЫЙ АНАЛИЗ НЕПОЛНЫХ ДУБЛИКАТОВ ПО ID ===

[clients.csv] Строк с конфликтующими client_id: 26
Уникальных client_id среди них: 13

Примеры конфликтующих записей клиентов (сравните строки с одинаковым client_id):


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
2373,C000122,1975-09-25,Ж,Новосибирск,Премиальный,Базовый,Кредитная,20.06.2019,26.04.2021,Не указан,55600.0,Нет,NaN,3.0,650,Да
2444,C000122,1975-09-25,Ж,ЕКАТЕРИНБУРГ,Массовый,Базовый,Кредитная,20.06.2019,26.04.2021,Не указан,55600.0,Нет,NaN,3.0,650,Да
230,C000166,1952-09-04,Ж,Нижний Новгород,Премиальный,Серебро,Кредитная,01.07.2016,03.01.2021,Зарплата в нашем банке,83700.0,Да,2024-12,NaN,106,Да
5016,C000166,1952-09-04,Ж,г. Москва,Зарплатный,Серебро,Кредитная,01.07.2016,03.01.2021,Зарплата в нашем банке,83700.0,Да,2024-12,NaN,106,Да
4506,C000183,1984-01-12,Ж,Москва,Зарплатный,Базовый,Дебетовая,06.12.2021,21.04.2022,Зарплата в нашем банке,54000.0,Да,2024-06,2.0,854,Нет
3889,C000183,1984-01-12,Ж,Екатеринбург,Private,Базовый,Дебетовая,06.12.2021,21.04.2022,Зарплата в нашем банке,54000.0,Да,2024-06,2.0,854,Нет
2745,C001680,1973-12-13,М,Екатеринбург,Массовый,Базовый,Дебетовая,25.11.2011,01.05.2016,Зарплата в другом банке,69500.0,Да,2024-06,3.0,306,Нет
3412,C001680,1973-12-13,М,Москва,Зарплатный,Базовый,Дебетовая,25.11.2011,01.05.2016,Зарплата в другом банке,69500.0,Да,2024-06,3.0,306,Нет
725,C002365,1981-06-23,Ж,Москва,Массовый,Серебро,Дебетовая,17.08.2016,23.11.2020,Не указан,30700.0,Да,2024-05,NaN,458,Да
3723,C002365,1981-06-23,Ж,МОСКВА,Массовый,Серебро,Дебетовая,17.08.2016,23.11.2020,Не указан,30700.0,Да,2024-05,NaN,458,Да



[transactions.csv] Строк с конфликтующими txn_id: 442
Уникальных txn_id среди них: 221

Примеры конфликтующих транзакций (первые 6 строк):


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
5066,105052,C002523,2024-01-06 13:43:43,"3358,1",RUB,Одежда и обувь,5651,СБП,успешно
5067,105052,C002523,2024-01-06,"3358,1",RUB,Одежда и обувь,5651,СБП,успешно
7681,107662,C001229,2024-01-08,"1578,1",RUB,АЗС,5541,Онлайн-банк,успешно
7682,107662,C001229,2024-01-08 23:22:30,"1578,1",RUB,АЗС,5541,Онлайн-банк,успешно
7836,107816,C001710,2024-01-09,"1222,81",RUB,Кафе и рестораны,5812,Токен,успешно
7837,107816,C001710,2024-01-09 10:38:33,"1222,81",RUB,Кафе и рестораны,5812,Токен,успешно
20575,120532,C004021,2024-01-23 15:09:00,"2263,41",RUB,Аптеки,5912,Карта,успешно
20576,120532,C004021,2024-01-23,"2263,41",RUB,Аптеки,5912,Карта,успешно
21644,121599,C000586,2024-01-24,"786,08",RUB,АЗС,5541,СБП,успешно
21645,121599,C000586,2024-01-24 22:32:19,"786,08",RUB,АЗС,5541,СБП,успешно


### 2.3. Соответствие данных описанию полей из задания

In [30]:
print("=" * 70)
print("=== 2.3. ПОЛНЫЙ ДИАГНОСТИЧЕСКИЙ АУДИТ ПОЛЕЙ И БИЗНЕС-ПРАВИЛ ===")
print("=" * 70)

# Вспомогательная дата среза по ТЗ (действующие клиенты на 30.06.2025)
SNAPSHOT_DATE = pd.to_datetime("2025-06-30")

# =========================================================================
# БЛОК 1. КЛИЕНТЫ: ДАТЫ И ВОЗРАСТ (Пункты 10, 11 и человеческий возраст)
# =========================================================================
print("\n>>> 1. АНАЛИЗ ДАТ И ВОЗРАСТА КЛИЕНТОВ")

# Временный парсинг дат для проверки
b_dates = pd.to_datetime(df_clients_raw['birth_date'], errors='coerce')
r_dates = pd.to_datetime(df_clients_raw['registration_date'], format='%d.%m.%Y', errors='coerce')
c_dates = pd.to_datetime(df_clients_raw['card_open_date'], format='%d.%m.%Y', errors='coerce')

# Расчет возраста на дату среза
ages = (SNAPSHOT_DATE - b_dates).dt.days // 365.25

print(f"Возраст клиентов (лет): min = {ages.min():.1f}, median = {ages.median():.1f}, max = {ages.max():.1f}")
unrealistic_age = ((ages < 14) | (ages > 100)).sum()
print(f"Неправдоподобный возраст (< 14 или > 100 лет): {unrealistic_age} из {len(df_clients_raw)}")

# Логика хронологии: Рождение -> Регистрация в банке -> Открытие карты -> Срез (30.06.2025)
future_reg = (r_dates > SNAPSHOT_DATE).sum()
future_card = (c_dates > SNAPSHOT_DATE).sum()
reg_before_birth = (r_dates < b_dates).sum()
card_before_reg = (c_dates < r_dates).sum()

print(f"Регистраций 'из будущего' (> 30.06.2025): {future_reg}")
print(f"Открытий карт 'из будущего' (> 30.06.2025): {future_card}")
print(f"Регистрация в банке раньше рождения: {reg_before_birth}")
print(f"Карта открыта раньше даты регистрации в банке: {card_before_reg}")


# =========================================================================
# БЛОК 2. КЛИЕНТЫ: ГОРОДА И ТЕКСТ (Пункт 8)
# =========================================================================
print("\n>>> 2. РАЗНОБОЙ В ГОРОДАХ (city)")
city_counts = df_clients_raw['city'].value_counts(dropna=False)
print("Все варианты написания городов и их частота:")
for city, cnt in city_counts.items():
    print(f"  - '{city}': {cnt}")


# =========================================================================
# БЛОК 3. КЛИЕНТЫ: ДОХОДЫ И ИХ ИСТОЧНИКИ (Пункты 12, 13, 17)
# =========================================================================
print("\n>>> 3. ДОХОДЫ И ИСТОЧНИКИ (monthly_income, income_source)")
print("Источники доходов:")
print(df_clients_raw['income_source'].value_counts(dropna=False))

inc = df_clients_raw['monthly_income']
print(f"\nАномальный доход (<0): {(inc < 0).sum()} из {len(df_clients_raw)}")
print(f"Нулевой доход (== 0): {(inc == 0).sum()} из {len(df_clients_raw)}")
print(f"Пропущенный доход (NaN): {inc.isna().sum()} из {len(df_clients_raw)}")
valid_inc = inc[inc > 0]
print("Статистика по положительным доходам:")
print(f"  min = {valid_inc.min():,.0f}, median = {valid_inc.median():,.0f}, 99% квантиль = {valid_inc.quantile(0.99):,.0f}, max = {valid_inc.max():,.0f}")
agr_to_adv = df_clients_raw['marketing_opt_in'].value_counts(dropna=False)
print("Количества согласных и несогласных с персональной рекламой")
for opt, count in agr_to_adv.items():
    print(f"{opt} - {count}")


# =========================================================================
# БЛОК 4. КЛИЕНТЫ: МОБИЛЬНОЕ ПРИЛОЖЕНИЕ И ОЦЕНКИ (Пункт 16)
# =========================================================================
print("\n>>> 4. МОБИЛЬНОЕ ПРИЛОЖЕНИЕ И УДОВЛЕТВОРЕННОСТЬ")
# Логика: если приложения нет, то дата входа должна быть пустой
no_app_with_login = ((df_clients_raw['has_mobile_app'] == 'Нет') & (df_clients_raw['app_last_login'].notna())).sum()
print(f"Клиентов БЕЗ приложения, но С датой последнего входа: {no_app_with_login}")

has_app_no_login = ((df_clients_raw['has_mobile_app'] == 'Да') & (df_clients_raw['app_last_login'].isna())).sum()
print(f"Клиентов С приложением, но БЕЗ даты последнего входа: {has_app_no_login}")

print("Пропуски в satisfaction_score:", df_clients_raw['satisfaction_score'].isna().sum(), f"({df_clients_raw['satisfaction_score'].isna().mean():.2%})")


# =========================================================================
# БЛОК 5. ТРАНЗАКЦИИ: СООТВЕТСТВИЕ MCC И КАТЕГОРИЙ (Пункты 9, 15)
# =========================================================================
print("\n>>> 5. КАТЕГОРИИ И MCC-КОДЫ")
mcc_cat_mapping = df_txn_raw.groupby('mcc_code')['category'].unique()
print("Соответствие MCC -> Категории:")
for mcc, cats in mcc_cat_mapping.items():
    print(f"  MCC {mcc}: {list(cats)}")

# Проверка: есть ли категории, у которых несколько разных MCC
cat_mcc_mapping = df_txn_raw.groupby('category')['mcc_code'].unique()
print("\nКатегории с несколькими разными MCC (если есть):")
mult_mcc = {cat: list(mccs) for cat, mccs in cat_mcc_mapping.items() if len(mccs) > 1}
print(mult_mcc if mult_mcc else "Каждому варианту категории соответствует ровно один MCC-код.")


# =========================================================================
# БЛОК 6. ТРАНЗАКЦИИ: СУММЫ, ВАЛЮТЫ И СТАТУСЫ (Пункты 14, 17)
# =========================================================================
print("\n>>> 6. СУММЫ, ВАЛЮТЫ И СТАТУСЫ ОПЕРАЦИЙ")
# Временная конвертация amount (замена запятой на точку)
amt_numeric = pd.to_numeric(df_txn_raw['amount'].astype(str).str.replace(',', '.'), errors='coerce')

print("Валюты по статусам операций:")
print(pd.crosstab(df_txn_raw['currency'], df_txn_raw['status'], margins=True))

print(f"\nСуммы: min = {amt_numeric.min()}, max = {amt_numeric.max()}")
print(f"Отрицательные суммы: {(amt_numeric < 0).sum()}")
print(f"Нулевые суммы: {(amt_numeric == 0).sum()}")
print(f"Сверхкрупные суммы (> 500 000 руб): {(amt_numeric > 500000).sum()}")

# Как оформлены возвраты: положительной суммой со статусом 'возврат' или отрицательной суммой?
refunds = df_txn_raw[df_txn_raw['status'] == 'возврат']
refused = df_txn_raw[df_txn_raw['status'] == 'отклонено']
accepted = df_txn_raw[df_txn_raw['status'] == 'успешно']
print(f"Минимальная и максимальная суммы возврата: {pd.to_numeric(refunds['amount'].str.replace(',', '.'), errors='coerce').min():.2f}, {pd.to_numeric(refunds['amount'].str.replace(',', '.'), errors='coerce').max():.2f}")
print(f"Минимальная и максимальная суммы отклонения: {pd.to_numeric(refused['amount'].str.replace(',', '.'), errors='coerce').min():.2f}, {pd.to_numeric(refused['amount'].str.replace(',', '.'), errors='coerce').max():.2f}")
print(f"Минимальная и максимальная суммы успешные: {pd.to_numeric(accepted['amount'].str.replace(',', '.'), errors='coerce').min():.2f}, {pd.to_numeric(accepted['amount'].str.replace(',', '.'), errors='coerce').max():.2f}")


# =========================================================================
# БЛОК 7. ТРАНЗАКЦИИ: ДАТЫ ОПЕРАЦИЙ И РЕГИСТРАЦИИ (Пункт 1, 10)
# =========================================================================
print("\n>>> 7. ДАТЫ ТРАНЗАКЦИЙ")
txn_dates = pd.to_datetime(df_txn_raw['txn_datetime'].str[:10], errors='coerce')
print(f"Транзакции раньше 2024-01-01: {(txn_dates < '2024-01-01').sum()} (min date: {txn_dates.min().strftime('%Y-%m-%d')})")
print(f"Транзакции позже 2025-06-30: {(txn_dates > '2025-06-30').sum()} (max date: {txn_dates.max().strftime('%Y-%m-%d')})")

=== 2.3. ПОЛНЫЙ ДИАГНОСТИЧЕСКИЙ АУДИТ ПОЛЕЙ И БИЗНЕС-ПРАВИЛ ===

>>> 1. АНАЛИЗ ДАТ И ВОЗРАСТА КЛИЕНТОВ
Возраст клиентов (лет): min = 22.0, median = 51.0, max = 88.0
Неправдоподобный возраст (< 14 или > 100 лет): 0 из 5693
Регистраций 'из будущего' (> 30.06.2025): 0
Открытий карт 'из будущего' (> 30.06.2025): 0
Регистрация в банке раньше рождения: 0
Карта открыта раньше даты регистрации в банке: 0

>>> 2. РАЗНОБОЙ В ГОРОДАХ (city)
Все варианты написания городов и их частота:
  - 'Москва': 1084
  - 'Нижний Новгород': 730
  - 'Санкт-Петербург': 657
  - 'Казань': 570
  - 'Новосибирск': 549
  - 'Екатеринбург': 461
  - 'Н. Новгород': 234
  - ' москва': 176
  - 'МОСКВА': 176
  - 'г. Москва': 166
  - 'казань': 153
  - 'г. Новосибирск': 147
  - 'ЕКАТЕРИНБУРГ': 125
  - 'г. Санкт-Петербург': 120
  - 'Санкт Петербург': 120
  - 'СПб': 117
  - 'г.Екатеринбург': 108

>>> 3. ДОХОДЫ И ИСТОЧНИКИ (monthly_income, income_source)
Источники доходов:
income_source
Зарплата в нашем банке     2559
Зарплата в д

In [31]:
print("=== 2.3. СООТВЕТСТВИЕ ОПИСАНИЮ ПОЛЕЙ ===\n")

print("--- 1. Категориальные поля clients ---")
for col in ['gender', 'segment', 'loyalty_tier', 'card_type', 'has_mobile_app', 'marketing_opt_in']:
    unique_vals = df_clients_raw[col].dropna().unique().tolist()
    print(f"{col} (уникальных: {len(unique_vals)}): {unique_vals}")

print("\n--- 2. Города (первые 20 и общее число) ---")
cities = df_clients_raw['city'].dropna().unique().tolist()
print(f"Всего уникальных написаний городов: {len(cities)}")
print("Примеры городов:", sorted(cities)[:15])

print("\n--- 3. Числовые поля clients ---")
print("satisfaction_score (уникальные значения):", sorted(df_clients_raw['satisfaction_score'].unique().tolist()))
print(f"monthly_income: min = {df_clients_raw['monthly_income'].min()}, max = {df_clients_raw['monthly_income'].max()}")

print("\n--- 4. Категориальные поля transactions ---")
for col in ['currency', 'channel', 'status']:
    print(f"{col}: {df_txn_raw[col].dropna().unique().tolist()}")

print("\nКатегории трат (category):")
categories = df_txn_raw['category'].dropna().unique().tolist()
print(f"Всего вариантов написания категорий: {len(categories)} (по ТЗ должно быть ровно 9)")
print("Варианты:", categories)

print("\n--- 5. Диапазон дат в транзакциях ---")
# Временный срез первых 10 символов (YYYY-MM-DD) для оценки диапазона
dates_sample = df_txn_raw['txn_datetime'].str[:10]
min_date, max_date = dates_sample.min(), dates_sample.max()
print(f"Фактический диапазон дат в transactions: от {min_date} до {max_date}")
print(f"Ожидаемый диапазон по ТЗ: от 2024-01-01 до 2025-06-30")

# Сколько транзакций выходят за рамки ТЗ
out_of_bounds = (~dates_sample.between("2024-01-01", "2025-06-30")).sum()
print(f"Транзакций вне периода исследования: {out_of_bounds} из {len(df_txn_raw)} ({out_of_bounds/len(df_txn_raw):.4%})")

=== 2.3. СООТВЕТСТВИЕ ОПИСАНИЮ ПОЛЕЙ ===

--- 1. Категориальные поля clients ---
gender (уникальных: 2): ['М', 'Ж']
segment (уникальных: 4): ['Массовый', 'Зарплатный', 'Премиальный', 'Private']
loyalty_tier (уникальных: 4): ['Серебро', 'Золото', 'Базовый', 'Платина']
card_type (уникальных: 3): ['Дебетовая', 'Кредитная', 'Премиальная']
has_mobile_app (уникальных: 2): ['Да', 'Нет']
marketing_opt_in (уникальных: 2): ['Нет', 'Да']

--- 2. Города (первые 20 и общее число) ---
Всего уникальных написаний городов: 17
Примеры городов: [' москва', 'ЕКАТЕРИНБУРГ', 'Екатеринбург', 'Казань', 'МОСКВА', 'Москва', 'Н. Новгород', 'Нижний Новгород', 'Новосибирск', 'СПб', 'Санкт Петербург', 'Санкт-Петербург', 'г. Москва', 'г. Новосибирск', 'г. Санкт-Петербург']

--- 3. Числовые поля clients ---
satisfaction_score (уникальные значения): [nan, 1.0, 2.0, 3.0, 4.0, 5.0]
monthly_income: min = -999.0, max = 1026300.0

--- 4. Категориальные поля transactions ---
currency: ['RUB', 'USD', 'EUR']
channel: ['СБП', 

### 2.4. Связь между таблицами

In [32]:
print("=== 2.4. СВЯЗЬ МЕЖДУ ТАБЛИЦАМИ (РЕФЕРЕНЦИАЛЬНАЯ ЦЕЛОСТНОСТЬ) ===")

set_clients_in_dict = set(df_clients_raw['client_id'])
set_clients_in_txn = set(df_txn_raw['client_id'])

# 1. Клиенты, у которых есть транзакции, но их нет в справочнике клиентов
orphan_txns_clients = set_clients_in_txn - set_clients_in_dict
orphan_txns_count = df_txn_raw['client_id'].isin(orphan_txns_clients).sum()
print(f"Транзакций от 'клиентов-призраков' (нет в clients.csv): {orphan_txns_count} из {len(df_txn_raw)} ({orphan_txns_count/len(df_txn_raw):.4%})")
print(f"Уникальных 'клиентов-призраков': {len(orphan_txns_clients)}")

# 2. Клиенты из справочника, у которых ВООБЩЕ нет ни одной транзакции
clients_without_txns = set_clients_in_dict - set_clients_in_txn
print(f"\nКлиентов без единой транзакции: {len(clients_without_txns)} из {len(df_clients_raw)} ({len(clients_without_txns)/len(df_clients_raw):.4%})")

=== 2.4. СВЯЗЬ МЕЖДУ ТАБЛИЦАМИ (РЕФЕРЕНЦИАЛЬНАЯ ЦЕЛОСТНОСТЬ) ===
Транзакций от 'клиентов-призраков' (нет в clients.csv): 30810 из 628754 (4.9002%)
Уникальных 'клиентов-призраков': 362

Клиентов без единой транзакции: 11 из 5693 (0.1932%)


**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

### Выводы по аудиту качества данных

В ходе аудита обнаружены следующие расхождения и дефекты данных:

1. **Нарушение уникальности первичных ключей и дубликаты:**
   - В `clients.csv` обнаружено 55 неуникальных `client_id` (0.97% от 5 693 строк), из них 42 — полные дубликаты строк, а 13 — записи с одинаковым ID, требующие отдельного анализа.
   - В `transactions.csv` обнаружено 1 500 неуникальных `txn_id` (0.24% от 628 754 строк), из них 1 279 — полные технические дубликаты строк (вероятно, повторные выгрузки пакетов данных).

2. **Ссылочная целостность (Foreign Keys):**
   - 30 810 транзакций (4.90% от 628 754) совершены 362 «клиентами-призраками», отсутствующими в `clients.csv`. Поскольку справочник содержит только действующих клиентов на 30.06.2025, эти клиенты, вероятнее всего, закрыли счета до даты среза. Для кампании по удержанию действующих клиентов эти операции нерелевантны.
   - 11 действующих клиентов (0.19% от 5 693) не совершили ни одной операции за весь период (новые или «мертвые» карты).

3. **Несоответствие форматов и бизнес-описания:**
   - Поле `amount` в транзакциях выгружено как текст с десятичной запятой (требует конвертации во float).
   - В валютах (`currency`) помимо RUB присутствуют USD и EUR.
   - Категории трат (`category`): 12 вариантов написания вместо 9 по ТЗ из-за регистра («Связь и интернет» / «Связь и Интернет») и символов («Кафе и рестораны» / «Кафе/рестораны»).
   - Города (`city`): 17 вариантов написания вместо реальных городов из-за регистра, приставок «г.» и сокращений («СПб», «Н. Новгород»).
   - Аномалии значений: доход `monthly_income` содержит отрицательные значения (-999.0), что является кодом отсутствия данных (маркером пропуска).
   - Временные границы: 76 транзакций (0.01%) выходят за пределы исследуемого периода (включая транзакции из будущего — до августа 2026 года).

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

In [33]:
# 1. Создаем рабочие копии
df_clients = df_clients_raw.copy()
df_txn = df_txn_raw.copy()

# =========================================================================
# ДЕДУПЛИКАЦИЯ ТРАНЗАКЦИЙ
# =========================================================================
# Удаляем полные дубликаты
n_full_dup_txn = df_txn.duplicated().sum()
df_txn = df_txn.drop_duplicates().reset_index(drop=True)

# Неполные дубли: сортируем так, чтобы строки с полным временем (длина строки > 10) были первыми
df_txn['dt_len'] = df_txn['txn_datetime'].astype(str).str.len()
df_txn = df_txn.sort_values(by=['txn_id', 'dt_len'], ascending=[True, False])
n_partial_dup_txn = df_txn.duplicated(subset=['txn_id']).sum()
df_txn = df_txn.drop_duplicates(subset=['txn_id'], keep='first').drop(columns=['dt_len']).reset_index(drop=True)

# Преобразование типов в transactions
df_txn['amount'] = df_txn['amount'].astype(str).str.replace(',', '.').astype(float)
df_txn['txn_datetime'] = pd.to_datetime(df_txn['txn_datetime'], format='mixed')

# =========================================================================
# ДЕДУПЛИКАЦИЯ КЛИЕНТОВ
# =========================================================================
# Удаляем полные дубликаты
n_full_dup_clients = df_clients.duplicated().sum()
df_clients = df_clients.drop_duplicates().reset_index(drop=True)

# Сортируем по иерархии сегментов: Private (0) > Премиальный (1) > Зарплатный (2) > Массовый (3)
segment_priority = {'Private': 0, 'Премиальный': 1, 'Зарплатный': 2, 'Массовый': 3}
df_clients['seg_rank'] = df_clients['segment'].map(segment_priority).fillna(99)
df_clients = df_clients.sort_values(by=['client_id', 'seg_rank'], ascending=[True, True])
n_partial_dup_clients = df_clients.duplicated(subset=['client_id']).sum()
df_clients = df_clients.drop_duplicates(subset=['client_id'], keep='first').drop(columns=['seg_rank']).reset_index(drop=True)

# Преобразование типов в clients
df_clients['birth_date'] = pd.to_datetime(df_clients['birth_date'], format='mixed')
df_clients['registration_date'] = pd.to_datetime(df_clients['registration_date'], format='mixed', dayfirst=True)
df_clients['card_open_date'] = pd.to_datetime(df_clients['card_open_date'], format='mixed', dayfirst=True)
df_clients['app_last_login'] = pd.to_datetime(df_clients['app_last_login'], format='mixed')
df_clients['satisfaction_score'] = df_clients['satisfaction_score'].astype('Int64')

# Обработка маркера пропуска -999 в доходе
n_neg_inc = (df_clients['monthly_income'] < 0).sum()
df_clients.loc[df_clients['monthly_income'] < 0, 'monthly_income'] = np.nan

print("=== ИТОГИ 3.1: ДЕДУПЛИКАЦИЯ И ПРИВЕДЕНИЕ ТИПОВ ===")
print(f"Транзакции: удалено полных дублей = {n_full_dup_txn}, усеченных по времени дублей = {n_partial_dup_txn}")
print(f"Итого строк transactions: {len(df_txn)} (все txn_id уникальны: {df_txn['txn_id'].is_unique})")
print(f"\nКлиенты: удалено полных дублей = {n_full_dup_clients}, конфликтующих по сегменту дублей = {n_partial_dup_clients}")
print(f"Итого строк clients: {len(df_clients)} (все client_id уникальны: {df_clients['client_id'].is_unique})")
print(f"Заменено аномальных доходов (-999 -> NaN): {n_neg_inc}")

=== ИТОГИ 3.1: ДЕДУПЛИКАЦИЯ И ПРИВЕДЕНИЕ ТИПОВ ===
Транзакции: удалено полных дублей = 1279, усеченных по времени дублей = 221
Итого строк transactions: 627254 (все txn_id уникальны: True)

Клиенты: удалено полных дублей = 42, конфликтующих по сегменту дублей = 13
Итого строк clients: 5638 (все client_id уникальны: True)
Заменено аномальных доходов (-999 -> NaN): 40


### 3.2. Категориальные признаки

In [34]:
# 1. Приведение 17 вариантов написания городов к 6 чистым городам
city_mapping = {
    'Москва': 'Москва', 'москва': 'Москва', 'МОСКВА': 'Москва', 'г. Москва': 'Москва',
    'Санкт-Петербург': 'Санкт-Петербург', 'г. Санкт-Петербург': 'Санкт-Петербург', 
    'Санкт Петербург': 'Санкт-Петербург', 'СПб': 'Санкт-Петербург',
    'Нижний Новгород': 'Нижний Новгород', 'Н. Новгород': 'Нижний Новгород',
    'Новосибирск': 'Новосибирск', 'г. Новосибирск': 'Новосибирск',
    'Казань': 'Казань', 'казань': 'Казань',
    'Екатеринбург': 'Екатеринбург', 'ЕКАТЕРИНБУРГ': 'Екатеринбург', 'г.Екатеринбург': 'Екатеринбург'
}
df_clients['city'] = df_clients['city'].str.strip().map(city_mapping).fillna(df_clients['city'])

# 2. Нормализация категорий транзакций
category_mapping = {
    'Связь и интернет': 'Связь и интернет',
    'Связь и Интернет': 'Связь и интернет',
    'Кафе и рестораны': 'Кафе и рестораны',
    'кафе и рестораны': 'Кафе и рестораны',
    'Кафе/рестораны': 'Кафе и рестораны',
    'Продукты': 'Продукты',
    'Одежда и обувь': 'Одежда и обувь',
    'Онлайн-сервисы': 'Онлайн-сервисы',
    'Аптеки': 'Аптеки',
    'Развлечения и досуг': 'Развлечения и досуг',
    'АЗС': 'АЗС',
    'Транспорт': 'Транспорт'
}
df_txn['category'] = df_txn['category'].map(category_mapping)

print("=== ИТОГИ 3.2: НОРМАЛИЗАЦИЯ КАТЕГОРИЙ ===")
print("Города в clients (ровно 6):", sorted(df_clients['city'].unique()))
print(f"Категории в transactions (ровно {df_txn['category'].nunique()}):", sorted(df_txn['category'].unique()))

=== ИТОГИ 3.2: НОРМАЛИЗАЦИЯ КАТЕГОРИЙ ===
Города в clients (ровно 6): ['Екатеринбург', 'Казань', 'Москва', 'Нижний Новгород', 'Новосибирск', 'Санкт-Петербург']
Категории в transactions (ровно 9): ['АЗС', 'Аптеки', 'Кафе и рестораны', 'Одежда и обувь', 'Онлайн-сервисы', 'Продукты', 'Развлечения и досуг', 'Связь и интернет', 'Транспорт']


### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

In [ ]:
print("=== 3.3. АУДИТ И РЕШЕНИЕ ПО ПРОПУСКАМ ===\n")

# 1. Пропуски в clients
client_nulls = df_clients.isna().sum()
client_nulls = client_nulls[client_nulls > 0]
for col, cnt in client_nulls.items():
    pct = cnt / len(df_clients)
    print(f"Поле '{col}': {cnt} пропусков из {len(df_clients)} ({pct:.2%})")
no_app_count = (df_clients['has_mobile_app'] == 'Нет').sum()
print(f"Людей без приложения {no_app_count}")


# 2. Пропуски в transactions
txn_nulls = df_txn.isna().sum()
print(f"\nПропусков в transactions: {txn_nulls.sum()}")

print("\nОбоснование решений по пропускам:")
print("1. 'monthly_income': пропущено у 562 клиентов из 5638. Механизм: MAR (клиент не указал доход добровольно).")
print("   Решение: Оставить NaN. Для группировки клиентов по доходу выделяется категория 'Доход не указан'. Заполнять медианой нельзя, чтобы не искажать распределение богатства.")
print("2. 'satisfaction_score': пропущено у 3 330 клиентов из 5638. Механизм: MNAR (опрос удовлетворенности добровольный).")
print("   Решение: Оставить NaN. Клиенты без ответа — это не 'нейтральные' клиенты, заполнение исказит результаты клиентского опыта.")
print("3. 'app_last_login': пропущено ровно у клиентов с has_mobile_app == 'Нет'.")
print("   Решение: Естественный пропуск (структурный ноль), заполнение не требуется.")

=== 3.3. АУДИТ И РЕШЕНИЕ ПО ПРОПУСКАМ ===

Поле 'monthly_income': 562 пропусков из 5638 (9.97%)
Поле 'app_last_login': 2124 пропусков из 5638 (37.67%)
Поле 'satisfaction_score': 3330 пропусков из 5638 (59.06%)
Людей без приложения 2124

Пропусков в transactions: 0

Обоснование решений по пропускам:
1. 'monthly_income': пропущено у 562 клиентов из 5638. Механизм: MAR (клиент не указал доход добровольно).
   Решение: Оставить NaN. Для группировки клиентов по доходу выделяется категория 'Доход не указан'. Заполнять медианой нельзя, чтобы не искажать распределение богатства.
2. 'satisfaction_score': пропущено у 3 300 клиентов из 5638. Механизм: MNAR (опрос удовлетворенности добровольный).
   Решение: Оставить NaN. Клиенты без ответа — это не 'нейтральные' клиенты, заполнение исказит результаты клиентского опыта.
3. 'app_last_login': пропущено ровно у клиентов с has_mobile_app == 'Нет'.
   Решение: Естественный пропуск (структурный ноль), заполнение не требуется.


### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

In [ ]:
print("=" * 70)
print("=== 3.4. ОБНАРУЖЕНИЕ ВЫБРОСОВ И РЕШЕНИЯ ===")
print("=" * 70)

# =========================================================================
# ШАГ 1. ОБНАРУЖЕНИЕ (ДЕТЕКЦИЯ)
# =========================================================================

# 1. Доходы клиентов (monthly_income)
inc_clean = df_clients['monthly_income'].dropna()
q25_inc, q75_inc = inc_clean.quantile(0.25), inc_clean.quantile(0.75)
iqr_inc = q75_inc - q25_inc
upper_bound_inc = q75_inc + 1.5 * iqr_inc
outliers_inc = df_clients[df_clients['monthly_income'] > upper_bound_inc]

print(f"1. Доходы (monthly_income):")
print(f"   Граница IQR (Q3 + 1.5*IQR): {upper_bound_inc:,.0f} руб.")
print(f"   Клиентов выше границы IQR: {len(outliers_inc)} из {len(inc_clean)} ({len(outliers_inc)/len(inc_clean):.2%})")
print(f"   Маскимум зарплаты {df_clients['monthly_income'].max()} руб")
print(f"   Распределение выбросов по сегментам:")
print(outliers_inc['segment'].value_counts())

# 2. Суммы транзакций (amount)
# Анализируем успешные положительные траты
pos_amounts = df_txn[df_txn['amount'] > 0]['amount']
q25_amt, q75_amt = pos_amounts.quantile(0.25), pos_amounts.quantile(0.75)
iqr_amt = q75_amt - q25_amt
upper_bound_amt = q75_amt + 1.5 * iqr_amt
outliers_amt = df_txn[df_txn['amount'] > upper_bound_amt]

print(f"\n2. Суммы покупок (amount):")
print(f"   Граница IQR: {upper_bound_amt:,.2f} руб., Максимум: {pos_amounts.max():,.2f} руб.")
print(f"   Транзакций выше границы IQR: {len(outliers_amt)} из {len(pos_amounts)} ({len(outliers_amt)/len(pos_amounts):.2%})")

# 3. Возраст клиентов
ages_clean = (pd.to_datetime("2025-06-30") - df_clients['birth_date']).dt.days // 365.25
q25_age, q75_age = ages_clean.quantile(0.25), ages_clean.quantile(0.75)
upper_bound_age = q75_age + 1.5 * (q75_age - q25_age)
lower_bound_age = q25_age - 1.5 * (q75_age - q25_age)
print(f"\n3. Возраст: диапазон от {ages_clean.min():.0f} до {ages_clean.max():.0f} лет.")
print(f"   Выбросов по возрасту (вне [{lower_bound_age:.0f}, {upper_bound_age:.0f}]): {((ages_clean < lower_bound_age) | (ages_clean > upper_bound_age)).sum()}")

print("=" * 70)
print("=== ДЕТАЛЬНЫЙ РАЗБОР ТОПОВЫХ ДОХОДОВ КЛИЕНТОВ ===")
print("=" * 70)

# 1. Топ-15 самых богатых клиентов банка
cols_to_show = ['client_id', 'monthly_income', 'segment', 'loyalty_tier', 'card_type', 'city', 'income_source']
top_earners = df_clients.sort_values(by='monthly_income', ascending=False)[cols_to_show].head(15)

print("\n>>> ТОП-15 клиентов по заявленному доходу:")
display(top_earners)

# 2. Сводка доходов по сегментам клиентов
def p90(x): 
    return x.quantile(0.9)

inc_by_segment = df_clients.groupby('segment')['monthly_income'].agg(
    клиентов='count',
    минимум='min',
    медиана='median',
    перцентиль_90=p90,
    максимум='max'
).loc[['Массовый', 'Зарплатный', 'Премиальный', 'Private']]

print("\n>>> Доходы в разрезе сегментов (руб.):")
display(inc_by_segment.style.format('{:,.0f}'))

# 3. Сколько людей зарабатывают больше 500 000 руб. и кто они?
rich_500k = df_clients[df_clients['monthly_income'] >= 500_000]
print(f"\nКлиентов с доходом >= 500 000 руб.: {len(rich_500k)} чел.")
print(rich_500k['segment'].value_counts())

# =========================================================================
# ШАГ 2. ОБОСНОВАНИЕ РЕШЕНИЙ
# =========================================================================
print("\n" + "=" * 50)
print("РЕШЕНИЯ ПО ВЫБРОСАМ (ДЛЯ ЖУРНАЛА ИЗМЕНЕНИЙ):")
print("1. Доходы > границы IQR: ОСТАВИТЬ БЕЗ ИЗМЕНЕНИЙ.")
print("   Обоснование: почти все выбросы приходятся на сегменты 'Премиальный' и 'Private'. Это реальные состоятельные клиенты банка, их удаление исказит финансовый профиль целевой аудитории.")
print("2. Крупные транзакции (до 147 тыс. руб.): ОСТАВИТЬ БЕЗ ИЗМЕНЕНИЙ.")
print("   Обоснование: Для категорий 'Одежда и обувь', 'Развлечения' и покупок премиальных клиентов чеки в 50–140 тыс. руб. являются естественными и экономически обоснованными.")
print("3. Отрицательные суммы: ОСТАВИТЬ (это возвраты покупок, status == 'возврат').")

=== 3.4. ОБНАРУЖЕНИЕ ВЫБРОСОВ И РЕШЕНИЯ ===
1. Доходы (monthly_income):
   Граница IQR (Q3 + 1.5*IQR): 192,300 руб.
   Клиентов выше границы IQR: 392 из 5076 (7.72%)
   Маскимум зарплаты 1026300.0 руб
   Распределение выбросов по сегментам:
segment
Премиальный    187
Private        183
Зарплатный      17
Массовый         5
Name: count, dtype: int64

2. Суммы покупок (amount):
   Граница IQR: 4,976.20 руб., Максимум: 147,058.41 руб.
   Транзакций выше границы IQR: 50298 из 619903 (8.11%)

3. Возраст: диапазон от 22 до 88 лет.
   Выбросов по возрасту (вне [16, 88]): 0
=== ДЕТАЛЬНЫЙ РАЗБОР ТОПОВЫХ ДОХОДОВ КЛИЕНТОВ ===

>>> ТОП-15 клиентов по заявленному доходу:


,client_id,monthly_income,segment,loyalty_tier,card_type,city,income_source
4369,C004662,1026300.0,Private,Золото,Дебетовая,Санкт-Петербург,Не указан
1182,C001249,854900.0,Private,Золото,Премиальная,Москва,Зарплата в другом банке
2913,C003108,788800.0,Private,Платина,Премиальная,Москва,Зарплата в другом банке
264,C000281,781700.0,Private,Платина,Дебетовая,Москва,Зарплата в другом банке
3871,C004132,777400.0,Private,Платина,Премиальная,Москва,Зарплата в другом банке
4084,C004360,776600.0,Private,Платина,Премиальная,Москва,Зарплата в другом банке
740,C000783,763700.0,Private,Платина,Кредитная,Москва,Зарплата в другом банке
4711,C005020,760200.0,Private,Золото,Премиальная,Москва,Зарплата в другом банке
5368,C005718,710600.0,Private,Золото,Премиальная,Санкт-Петербург,Зарплата в нашем банке
1258,C001333,665300.0,Private,Платина,Кредитная,Санкт-Петербург,Зарплата в другом банке



>>> Доходы в разрезе сегментов (руб.):


,клиентов,минимум,медиана,перцентиль_90,максимум
segment,,,,,
Массовый,"2,496",24,"60,100","99,650","269,600"
Зарплатный,"1,593",32,"77,300","127,760","268,500"
Премиальный,786,83,"146,400","245,350","460,800"
Private,201,274,"349,000","560,100","1,026,300"



Клиентов с доходом >= 500 000 руб.: 33 чел.
segment
Private    33
Name: count, dtype: int64

РЕШЕНИЯ ПО ВЫБРОСАМ (ДЛЯ ЖУРНАЛА ИЗМЕНЕНИЙ):
1. Доходы > границы IQR: ОСТАВИТЬ БЕЗ ИЗМЕНЕНИЙ.
   Обоснование: 100% выбросов приходятся на сегменты 'Премиальный' и 'Private'. Это реальные состоятельные клиенты банка, их удаление исказит финансовый профиль целевой аудитории.
2. Крупные транзакции (до 147 тыс. руб.): ОСТАВИТЬ БЕЗ ИЗМЕНЕНИЙ.
   Обоснование: Для категорий 'Одежда и обувь', 'Развлечения' и покупок премиальных клиентов чеки в 50–140 тыс. руб. являются естественными и экономически обоснованными.
3. Отрицательные суммы: ОСТАВИТЬ (это возвраты покупок, status == 'возврат').


### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

In [ ]:
print("=== 3.5. ВОРОНКА ФИЛЬТРАЦИИ ДАННЫХ ПЕРЕД РАСЧЁТОМ АКТИВНОСТИ ===\n")

funnel_records = []
initial_txn_count = len(df_txn_raw)
funnel_records.append({"Шаг / Фильтр": "1. Исходные сырые транзакции", "Осталось строк": initial_txn_count, "Удалено": 0})

# Шаг 2. Дедупликация
txn_filtered = df_txn.copy()
dropped_dups = initial_txn_count - len(txn_filtered)
funnel_records.append({"Шаг / Фильтр": "2. Удаление технических дублей", "Осталось строк": len(txn_filtered), "Удалено": dropped_dups})

# Шаг 3. Окно дат (2024-01-01 <= dt <= 2025-06-30)
before_dates = len(txn_filtered)
date_mask = (txn_filtered['txn_datetime'] >= '2024-01-01') & (txn_filtered['txn_datetime'] <= '2025-06-30 23:59:59')
txn_filtered = txn_filtered[date_mask].reset_index(drop=True)
dropped_dates = before_dates - len(txn_filtered)
funnel_records.append({"Шаг / Фильтр": "3. Фильтр окна дат (2024-01-01 .. 2025-06-30)", "Осталось строк": len(txn_filtered), "Удалено": dropped_dates})

# Шаг 4. Исключение транзакций клиентов-призраков
before_clients = len(txn_filtered)
active_client_ids = set(df_clients['client_id'])
txn_filtered = txn_filtered[txn_filtered['client_id'].isin(active_client_ids)].reset_index(drop=True)
dropped_orphans = before_clients - len(txn_filtered)
funnel_records.append({"Шаг / Фильтр": "4. Исключение клиентов не из справочника действующей базы", "Осталось строк": len(txn_filtered), "Удалено": dropped_orphans})

# Шаг 5. Исключение отклоненных операций (status == 'отклонено')
before_declined = len(txn_filtered)
txn_filtered = txn_filtered[txn_filtered['status'] != 'отклонено'].reset_index(drop=True)
dropped_declined = before_declined - len(txn_filtered)
funnel_records.append({"Шаг / Фильтр": "5. Исключение несостоявшихся операций ('отклонено')", "Осталось строк": len(txn_filtered), "Удалено": dropped_declined})

# Шаг 6. Валютный фильтр (оставляем рублевые операции, так как бюджет и лимиты в RUB)
before_curr = len(txn_filtered)
txn_filtered = txn_filtered[txn_filtered['currency'] == 'RUB'].reset_index(drop=True)
dropped_curr = before_curr - len(txn_filtered)
funnel_records.append({"Шаг / Фильтр": "6. Исключение валютных операций (USD, EUR)", "Осталось строк": len(txn_filtered), "Удалено": dropped_curr})

# Сводная таблица воронки
df_funnel = pd.DataFrame(funnel_records)
df_funnel["Доля от исходных данных"] = (df_funnel["Осталось строк"] / initial_txn_count).map("{:.2%}".format)
df_funnel["Доля шага"] = (df_funnel["Удалено"] / initial_txn_count).map("{:.4%}".format)
display(df_funnel)

# Сохраняем итоговый рабочий датасет транзакций для анализа
df_txn_clean = txn_filtered.copy()
print(f"\nИтоговый объем очищенных валидных операций: {len(df_txn_clean)} строк")

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")